In [36]:
import requests
import pandas as pd
import time

API_KEY = "579b464db66ec23bdd000001c11cc22deb2d41c349c349def94e777e"
DATASET_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
BASE_URL = f"https://api.data.gov.in/resource/{DATASET_ID}?format=json&api-key={API_KEY}"

LIMIT = 100
MAX_ROWS = 5000

all_records = []
headers = {"User-Agent": "Mozilla/5.0"}

# Extract step: paginate through results
for offset in range(0, MAX_ROWS, LIMIT):
    url = f"{BASE_URL}&limit={LIMIT}&offset={offset}"
    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            data = response.json()
            all_records.extend(data.get('records', []))
        else:
            print(f"API error {response.status_code} at offset {offset}")
            time.sleep(5)
    except Exception as e:
        print("Error:", e)
        time.sleep(5)

# Convert to DataFrame
df = pd.DataFrame(all_records)

df.columns.to_list()
print("Min date:", df['Arrival_Date'].min())
print("Max date:", df['Arrival_Date'].max())

# Drop rows with missing Arrival_Date
df = df.dropna(subset=['Arrival_Date'])

# Parse Arrival_Date as datetime
df['Arrival_Date'] = pd.to_datetime(df['Arrival_Date'], errors='coerce')

# Drop rows that failed conversion
df = df.dropna(subset=['Arrival_Date'])

# Filter for 2023–2025
start_date = pd.to_datetime("2009-01-01")
end_date = pd.to_datetime("2009-12-31")

df_2009 = df[(df['Arrival_Date'] >= start_date) & (df['Arrival_Date'] <= end_date)]
#df = df[(df['Arrival_Date'].dt.year >= 2023) & (df['Arrival_Date'].dt.year <= 2025)]

print("Total 2009 records:", len(df_2009))
print("Date range:", df_2009['Arrival_Date'].min(), "to", df_2009['Arrival_Date'].max())

# Save filtered dataset
df_2009.to_csv("India_ArrivalDate_2009.csv", index=False)

print("Filtered dataset for 2009 based on Arrival_Date:")
print(df_2009.head())
#print("Columns:", df_2023.columns.tolist())


Min date: 01/01/2009
Max date: 31/12/2009
Total 2009 records: 1865
Date range: 2009-01-01 00:00:00 to 2009-12-12 00:00:00
Filtered dataset for 2009 based on Arrival_Date:
  Arrival_Date Commodity Commodity_Code District Grade       Market Max_Price  \
0   2009-11-11    Potato             24  Hooghly   FAQ  Sheoraphuly      1600   
2   2009-02-12    Potato             24  Hooghly   FAQ  Sheoraphuly      1660   
3   2009-03-12    Potato             24  Hooghly   FAQ  Sheoraphuly      1600   
4   2009-04-12    Potato             24  Hooghly   FAQ  Sheoraphuly      1600   
5   2009-05-12    Potato             24  Hooghly   FAQ  Sheoraphuly      1630   

  Min_Price Modal_Price        State Variety  
0      1580        1580  West Bengal   Jyoti  
2      1650        1650  West Bengal   Jyoti  
3      1580        1580  West Bengal   Jyoti  
4      1580        1580  West Bengal   Jyoti  
5      1620        1620  West Bengal   Jyoti  
